# 🧪 Project 11 | Agent fine-tuning with SFT and QLoRA

Choose a GPU runtime in Colab and run the cells in order. The reported verification used a Tesla T4. This notebook reconstructs the cells from that run. The install cell resolves the current Unsloth package and its dependencies.

Tutorial credit: [Darsh Shah, freeCodeCamp](https://www.freecodecamp.org/news/how-to-customize-an-llm-for-ai-agents-using-sft-and-qlora/).

In [ ]:
%pip install unsloth

If Colab prompts you to restart the session after installation, restart it and continue with the next cell.

In [ ]:
from unsloth import FastLanguageModel
import torch

base_checkpoint = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_checkpoint,
    max_seq_length=512,
    load_in_4bit=True,
)
print("Model device:", next(model.parameters()).device)
print("Loaded in 4-bit:", model.is_loaded_in_4bit)

In [ ]:
messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful ecommerce customer support assistant. "
            "Be polite, concise, and do not invent order details."
        ),
    },
    {"role": "user", "content": "I want to cancel my order."},
]

def generate_reply(model, tokenizer, messages):
    FastLanguageModel.for_inference(model)
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=80, do_sample=False)

    prompt_length = inputs["input_ids"].shape[-1]
    return tokenizer.decode(
        output[0, prompt_length:], skip_special_tokens=True
    ).strip()

baseline_reply = generate_reply(model, tokenizer, messages)
print("=== BEFORE SFT ===")
print(baseline_reply)

In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

support_examples = [
    (
        "Has my package shipped yet?",
        "Please share your order number so I can check the shipping status.",
    ),
    (
        "Can you change where my package is going?",
        "Please share your order number. If the package has already shipped, "
        "we may need to check what options the carrier offers.",
    ),
    (
        "The item arrived cracked.",
        "Please share your order number and a photo of the damage so we can "
        "review your replacement options.",
    ),
    (
        "My card shows two payments for one purchase.",
        "Please share your order number and the dates of the two charges "
        "so we can review them.",
    ),
    (
        "Can I swap this shirt for a larger size?",
        "Please share your order number and the size you need so we can "
        "check exchange eligibility and availability.",
    ),
]

train_dataset = Dataset.from_list([
    {"messages": [
        messages[0],
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]}
    for question, answer in support_examples
])

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    args=SFTConfig(
        output_dir="/content/project11_training",
        max_length=512,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=2,
        max_steps=30,
        learning_rate=2e-4,
        warmup_steps=5,
        fp16=True,
        bf16=False,
        optim="adamw_torch",
        logging_steps=5,
        save_strategy="no",
        report_to="none",
        seed=3407,
    ),
)

result = trainer.train()
adapter_dir = "/content/project11_qwen25_15b_adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"Training loss: {result.training_loss:.4f}")
print("Adapter saved:", adapter_dir)

In [ ]:
adapted_reply = generate_reply(model, tokenizer, messages)
print("=== BEFORE SFT ===")
print(baseline_reply)
print("\n=== AFTER SFT ===")
print(adapted_reply)

held_out_prompts = [
    "Tracking says delivered, but I never received my package.",
    "I canceled my order yesterday. Has my refund been issued?",
    "Can you guarantee that my order will arrive tomorrow?",
]

for prompt in held_out_prompts:
    case = [messages[0], {"role": "user", "content": prompt}]
    with model.disable_adapter():
        before = generate_reply(model, tokenizer, case)
    after = generate_reply(model, tokenizer, case)

    print(f"\nPROMPT: {prompt}")
    print(f"BEFORE: {before}")
    print(f"AFTER:  {after}")

In [ ]:
reloaded_model, reloaded_tokenizer = FastLanguageModel.from_pretrained(
    model_name=adapter_dir,
    max_seq_length=512,
    load_in_4bit=True,
)
reloaded_reply = generate_reply(reloaded_model, reloaded_tokenizer, messages)
print("=== RELOADED ADAPTER ===")
print(reloaded_reply)
print("Matches original adapted reply:", reloaded_reply == adapted_reply)

### 📦 Optional adapter download

Run this cell if you want to keep a local copy of the trained adapter and tokenizer.

In [ ]:
from shutil import make_archive
from google.colab import files

archive = make_archive("/content/project11_qwen25_15b_adapter", "zip", adapter_dir)
files.download(archive)